# MCP Math Server + LangChain Agent Demo

This notebook demonstrates the MCP architecture from the practical lesson:

- A reusable **Math MCP Server**
- A **LangChain MCP Client**
- Dynamic MCP tool discovery
- A LangChain agent using discovered MCP tools
- Multiple test cases
- Error handling for division by zero

> **Architecture:** User → LangChain Agent → MCP Client → MCP Server → Tool


## 1. Install Dependencies

Run this cell once in your notebook environment.


In [ ]:
%pip install -q fastmcp langchain langchain-openai langchain-mcp-adapters python-dotenv


## 2. Configure the OpenAI API Key

For a local notebook, you can use a `.env` file.

Example:

```text
OPENAI_API_KEY=your_openai_api_key_here
```

The code below loads the environment variables.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is not configured. "
        "Add it to your .env file or environment."
    )

print("OpenAI API Key Found")


## 3. Create the Math MCP Server

The MCP server contains the reusable tools.

Unlike the original LangChain agent, these tools are **not defined directly inside the agent code**.

The server exposes:

- `add`
- `multiply`
- `divide`
- `square_root`


In [ ]:
from fastmcp import FastMCP
import math

# Create the MCP server
mcp = FastMCP("Math MCP Server")


@mcp.tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b


@mcp.tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b


@mcp.tool
def divide(a: float, b: float) -> float:
    """Divide the first number by the second number."""
    if b == 0:
        raise ValueError("Cannot divide by 0")
    return a / b


@mcp.tool
def square_root(a: float) -> float:
    """Calculate the square root of a number."""
    if a < 0:
        raise ValueError(
            "Cannot calculate square root of a negative number"
        )
    return math.sqrt(a)

print("Math MCP Server created successfully.")


## 4. Save the MCP Server to a Python File

The local STDIO MCP architecture launches the server as a subprocess.

Therefore, save the MCP server implementation as:

`math_mcp_server.py`

This cell creates that file automatically from the notebook.


In [ ]:
server_code = '''
from fastmcp import FastMCP
import math

mcp = FastMCP("Math MCP Server")


@mcp.tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b


@mcp.tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b


@mcp.tool
def divide(a: float, b: float) -> float:
    """Divide the first number by the second number."""
    if b == 0:
        raise ValueError("Cannot divide by 0")
    return a / b


@mcp.tool
def square_root(a: float) -> float:
    """Calculate the square root of a number."""
    if a < 0:
        raise ValueError(
            "Cannot calculate square root of a negative number"
        )
    return math.sqrt(a)


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("math_mcp_server.py", "w", encoding="utf-8") as f:
    f.write(server_code.strip())

print("Created: math_mcp_server.py")


## 5. Create the MCP Client

The client connects to the local MCP server using:

- `transport = "stdio"`
- `command = "python"`
- The path to `math_mcp_server.py`

The server will be launched as a subprocess when the client connects.


In [ ]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient

server_path = os.path.abspath("math_mcp_server.py")

client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",
            "command": "python",
            "args": [server_path],
        }
    }
)

print("MCP client configured.")
print("Server:", server_path)


## 6. Discover MCP Tools

The MCP client asks the server for its available tools.

This is the dynamic discovery feature of MCP.

The agent does not need to hard-code the four math functions.


In [ ]:
import asyncio

async def discover_tools():
    tools = await client.get_tools()
    return tools

tools = await discover_tools()

print("Discovered MCP Tools:")
for tool in tools:
    print(f"- {tool.name}: {tool.description}")


## 7. Initialize the LLM

The LLM remains the reasoning engine.

MCP does not replace the LLM or the ReAct agent loop.
It provides a standardized way to discover and execute external tools.


In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gpt-5.5",
    model_provider="openai"
)

print("LLM initialized.")


## 8. Create the LangChain Agent

The discovered MCP tools are passed to the LangChain agent.

The agent can now reason about which MCP tool it needs to call.


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools
)

print("LangChain agent created with MCP tools.")


## 9. Run a Single Test

The agent should discover that the calculation requires the MCP `multiply` tool.


In [ ]:
async def run_agent(question):
    response = await agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )

    return response["messages"][-1].content


answer = await run_agent(
    "What is 15 multiplied by 8?"
)

print("Agent Answer:")
print(answer)


## 10. Run Multiple Test Cases

These examples demonstrate:

1. Simple addition
2. Multiplication
3. Square root
4. Multi-step reasoning
5. Division-by-zero error handling


In [ ]:
test_cases = [
    "What is 25 plus 15?",
    "What is 15 multiplied by 8?",
    "What is the square root of 144?",
    "What is the square root of the area of a rectangle with length 15 and width 8?",
    "What is 100 divided by 0?"
]


for index, question in enumerate(test_cases, start=1):

    print("\n" + "=" * 60)
    print(f"TEST CASE {index}")
    print("=" * 60)

    print("Question:", question)

    try:
        answer = await run_agent(question)
        print("\nAgent Answer:")
        print(answer)

    except Exception as e:
        print("\nAgent Error:")
        print(str(e))


## 11. Architecture

The final architecture looks like this:

```text
                 User
                   |
                   v
            LangChain Agent
                   |
                   v
                  LLM
                   |
             Tool Decision
                   |
                   v
              MCP Client
                   |
          JSON-RPC over STDIO
                   |
                   v
            MCP Server
                   |
          +--------+--------+
          |        |        |
        add    multiply   divide
                            |
                       square_root
```

### Key takeaway

Before MCP:

```text
Agent → Local Python Tool
```

After MCP:

```text
Agent → MCP Client → MCP Server → Tool
```

The main change is **where the tools live**.

The LLM and agent reasoning process remain conceptually the same, while MCP provides a standardized layer for tool discovery and execution.
